# Solar Filament Segmentation: download unlabelled GONG frames (CPU)

Downloads unlabelled H-alpha frames from days at least 2 days away from every training and test image (one frame per
competition site and day) into `/kaggle/working/unlabelled/` (`frames/*.jpg`, `manifest.csv`), with
`scripts/data/download_unlabelled.py`. The output is then attached to the GPU notebooks for pseudo-labelling.

### Settings
* Accelerator: **None** (CPU only: no GPU quota). Internet: **on**.
* *Add Input*: competition **filament-segmentation-2026** (to know the training and test dates).
* *Add-ons → Secrets*: the GitHub token (label `GITHUB_SECRET`). Run with **Save Version → Save & Run All**.

In [ ]:
# ---- configuration ---------------------------------------------------------------------------------------
DAYS = 600                             # days sampled evenly over 2011-2022 (~5 frames per day: ~3,000 frames)
MARGIN_DAYS = 2                        # minimum distance from every training and test image
GITHUB_SECRET = "Github_key"           # label of the Kaggle secret holding the GitHub token
WANDB_SECRET = "WANDB_API_KEY"         # (not needed here)
REPO_URL = "github.com/LucaT2/solar-filament-segmentation.git"
REPO_BRANCH = "main"
REPO_DATASET = None

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess, glob, json, time
from pathlib import Path

T0 = time.time()
def log(msg):   # timestamped progress, so a stalled step is easy to spot in the log
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - T0:5.0f}s] {msg}", flush=True)

log("session started")
# no step may wait for input: git never asks for a password (a rejected token fails instead of hanging)
os.environ["GIT_TERMINAL_PROMPT"] = "0"

def run(cmd, timeout, what):
    """Run a setup command; fail loudly (instead of hanging) after ``timeout`` seconds."""
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, stdin=subprocess.DEVNULL)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{what} did not finish within {timeout} s (network problem?)") from None
    return res

log("reading secrets")
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
# less GPU-memory fragmentation between training and full-image validation (inherited by all subprocesses)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
try:   # optional W&B tracking; training subprocesses inherit the environment variable
    os.environ["WANDB_API_KEY"] = secrets.get_secret(WANDB_SECRET)
    log("W&B tracking enabled")
except Exception:
    os.environ["WANDB_MODE"] = "disabled"
    log(f"no '{WANDB_SECRET}' secret: tracking disabled")

def github_token():
    try:
        return secrets.get_secret(GITHUB_SECRET)
    except Exception as exc:
        raise RuntimeError(
            f"Kaggle secret '{GITHUB_SECRET}' is not attached to this notebook. In the editor: Add-ons -> Secrets -> "
            f"make sure a secret with exactly this label exists and its checkbox is ticked, or change GITHUB_SECRET "
            f"in the configuration cell. Then Save Version -> Save & Run All again."
        ) from exc

def find_input(name, marker):
    """Locate an attached dataset by name (Kaggle's mount layout varies), via a file it must contain."""
    hits = [Path(p).parent for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True) if name in p]
    if not hits:
        raise FileNotFoundError(f"attached dataset '{name}' with {marker} not found under /kaggle/input")
    return min(hits, key=lambda p: len(str(p)))

REPO = Path("/kaggle/working/repo")
if REPO_DATASET:
    shutil.copytree(find_input(REPO_DATASET, "pyproject.toml"), REPO, dirs_exist_ok=True)
elif not REPO.exists():
    token = github_token()
    log("GitHub token read; cloning the repository")
    # token as the password (required for fine-grained tokens; also works for classic ones)
    res = run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH,
               f"https://x-access-token:{token}@{REPO_URL}", str(REPO)], 120, "git clone")
    if res.returncode != 0:   # token never printed
        raise RuntimeError(
            "git clone failed: " + res.stderr.replace(token, "***") +
            "\nCheck the GitHub token: not expired; 'Repository access' includes solar-filament-segmentation; "
            "'Repository permissions -> Contents: Read-only'.")
log("repo: " + subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Kaggle images ship torch, numpy, pandas, opencv, scikit-image, scikit-learn, timm and wandb. Install only what is
# missing, pinned to the versions tested locally (requirements.txt), without touching the preinstalled stack.
log("installing packages")
for cmd in ([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "segmentation-models-pytorch==0.5.0",
             "albumentations==2.0.8", "albucore==0.0.24", "pycocotools"],
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)]):
    res = run(cmd, 300, "pip install")
    if res.returncode != 0:
        raise RuntimeError("pip install failed:\n" + res.stderr[-2000:])
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import torch, yaml
log(f"packages installed; torch {torch.__version__}, "
    f"GPUs {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
!python scripts/setup/check_env.py
log("environment checked")

## 2. Download

In [ ]:
# ~40 min of archive listings, then ~2 h of downloads (one polite request per ~2 s); resumable
!python -W ignore scripts/data/download_unlabelled.py --days {DAYS} --margin-days {MARGIN_DAYS} --out /kaggle/working/unlabelled
import pandas as pd
m = pd.read_csv("/kaggle/working/unlabelled/manifest.csv")
log(f"{len(m)} frames, {int(m.ok.sum())} pass the quality check; per site: {m[m.ok].site.value_counts().to_dict()}")